# LLM chat comparison (no API)

The first two documents (one Spanish and one Catalan) were processed in the current Codex chat. The chat selected unique literal entity values for the seven labels. An exact-string alignment helper expands all occurrences and calculates their character offsets. This is a chat-plus-alignment experiment; the chat did not generate numeric offsets unaided.

The local response is `results/llm/chat_response.jsonl`. No entity text or source document is published.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'ner').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ner.evaluation import ENTITY_LABELS, evaluate_ner_strict, evaluate_ner_relaxed
from ner.utils import load_jsonl, save_responses_to_jsonl
from ner.llm import align_chat_response
gold = load_jsonl(ROOT / 'data/plain/validation_small.jsonl')
responses = load_jsonl(ROOT / 'results/llm/chat_response.jsonl')
predictions = align_chat_response(gold, responses)
output = ROOT / 'results/llm/chat.jsonl'
save_responses_to_jsonl(predictions, output)
print('Chat documents:', len(predictions))
strict = evaluate_ner_strict(ROOT / 'data/plain/validation_small.jsonl', output)
relaxed = evaluate_ner_relaxed(ROOT / 'data/plain/validation_small.jsonl', output)

Chat documents: 2

STRICT EVALUATION
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 82 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00      20       0       0
ID_CARD_NUMBER              1.00   1.00   1.00       8       0       0
ADDRESS                     1.00   1.00   1.00       8       0       0
REGISTRATION_NUMBER         1.00   1.00   1.00      10       0       0
DATE                        1.00   1.00   1.00      16       0       0
AMOUNT                      1.00   1.00   1.00       8       0       0
ORGANIZATION                1.00   1.00   1.00      12       0       0

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 82 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
---------------------------------------------------------------------

## Matched comparison

Restrict the GLiNER predictions to exactly the same two documents; the full 50-document scores are not a controlled comparison with this small chat sample.

In [2]:
ids = {s['id'] for s in gold}
for name in ['small', 'large']:
    gliner_predictions = [p for p in load_jsonl(ROOT / f'results/plain/gliner_{name}.jsonl') if p['id'] in ids]
    matched_path = ROOT / f'results/plain/gliner_{name}_chat_subset.jsonl'
    save_responses_to_jsonl(gliner_predictions, matched_path)
    print('Model:', name)
    strict = evaluate_ner_strict(ROOT / 'data/plain/validation_small.jsonl', matched_path)
    relaxed = evaluate_ner_relaxed(ROOT / 'data/plain/validation_small.jsonl', matched_path)

Model: small

STRICT EVALUATION
Micro -> P: 0.65 | R: 0.80 | F1: 0.72 | TP: 66 | FP: 36 | FN: 16

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.91   1.00   0.95      20       2       0
ID_CARD_NUMBER              1.00   0.75   0.86       6       0       2
ADDRESS                     0.07   0.12   0.09       1      13       7
REGISTRATION_NUMBER         0.77   1.00   0.87      10       3       0
DATE                        1.00   1.00   1.00      16       0       0
AMOUNT                      0.45   0.62   0.53       5       6       3
ORGANIZATION                0.40   0.67   0.50       8      12       4

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.65 | R: 0.80 | F1: 0.72 | TP: 66 | FP: 36 | FN: 16

ENTITY TYPE                    P      R     F1      TP      FP      FN
----------------------------------------------------------------------

Two documents cannot establish general model superiority. Exact-string alignment assists the chat with offsets; this difference must be considered when interpreting strict scores.